# FramePilot: which Whisper model understands Egyptian Arabic best?

This notebook tests **4 speech-to-text models** on **our own recordings** (from Google Drive) and on
**MGB-3** (Egyptian YouTube clips from the internet that already have the correct text).

**How to run**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Edit the **SETTINGS** cell (step 1) so the paths match your Google Drive.
3. Menu **Runtime → Run all**. Allow Google Drive access when asked.
4. Wait (about 15-25 minutes). The results table is at the end, and it is also saved to Drive.

**What the numbers mean**
- **WER** = word error rate = share of words that are wrong (0.30 = 30 % wrong). **Lower is better.**
  WER for our sessions needs a hand-corrected text file `ref_s01.txt` (see step 1); without it only MGB-3 has WER.
- **CER** = the same but counting letters (fairer for Arabic spelling differences). Lower is better.
- **confidence** = how sure Whisper is of its own words (0-1). Higher is usually better, but it is only a hint.
- **invented** = sentences Whisper made up (like "اشتركوا في القناة"). Must be 0.
- **raw / clean / clean+hints** = original audio / noise-filtered audio / filtered audio + Egyptian hint words
  (exactly what our pipeline does).

## Step 1 - SETTINGS (edit this cell)

In [ ]:
# ===== change these to match your Google Drive =====
DRIVE_SESSIONS = "/content/drive/MyDrive/FramePilot/data/sessions"  # the folder that has s01, s02, s03 inside
RESULTS = "/content/drive/MyDrive/FramePilot/asr_test"            # results are saved here (made if missing)
# Optional: put hand-corrected text files here as ref_s01.txt, ref_s03.txt (what was REALLY said
# between START and END seconds). Then you also get WER for our own recordings.

SESSIONS = ["s01", "s03"]     # Arabic sessions to test
START, END = 0, 120           # which part of each session, in seconds (120 = first 2 minutes)
MGB3_CLIPS = 40               # internet Egyptian clips with correct text (0 = skip that test)
KEEP_MODELS_ON_DRIVE = False  # True = keep converted models in RESULTS/models (~2.5 GB) -> next run is faster

# name in the table -> model.
#   "large-v3-turbo"            = ready for faster-whisper (downloads by itself)
#   ("hf id", "original model") = normal Hugging Face model: converted for faster-whisper automatically
MODELS = {
    "turbo (now)": "large-v3-turbo",
    "large-v3": "large-v3",
    "turbo-dialectal": ("oddadmix/whisper-large-v3-turbo-arabic-dialectal", "openai/whisper-large-v3-turbo"),
    "small-egy-codeswitch": ("IbrahimAmin/code-switched-egyptian-arabic-whisper-small", "openai/whisper-small"),
}

## Step 2 - install tools and check the GPU

In [ ]:
!nvidia-smi -L
!pip install -q faster-whisper jiwer datasets

In [ ]:
# faster-whisper needs the CUDA 12 version of cuBLAS (+ cuDNN 9). If Colab does not have them, add them.
import ctypes, glob, os, subprocess, sys

def can_load(lib):
    try:
        ctypes.CDLL(lib); return True
    except OSError:
        return False

for lib, pkg, mod in [("libcublas.so.12", "nvidia-cublas-cu12", "cublas"), ("libcudnn.so.9", "nvidia-cudnn-cu12==9.*", "cudnn")]:
    if not can_load(lib):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        import importlib; m = importlib.import_module("nvidia." + mod)
        for f in sorted(glob.glob(os.path.join(list(m.__path__)[0], "lib", "lib*.so*"))):
            ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)          # load it now so faster-whisper finds it
    print(lib, "OK")
import ctranslate2
print("GPUs faster-whisper can use:", ctranslate2.get_cuda_device_count(), "(must be 1)")

## Step 3 - connect Google Drive and check the files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
for s in SESSIONS:
    folder = Path(DRIVE_SESSIONS) / s
    missing = [f for f in ["audio.wav", "session.json"] if not (folder / f).exists()]
    if missing:
        raise FileNotFoundError(f"{folder} has no {missing}. Fix DRIVE_SESSIONS in step 1.")
    print(s, "OK")
Path(RESULTS).mkdir(parents=True, exist_ok=True)

## Step 4 - our pipeline's settings
Copied from `ai_director/audio/asr.py` and `eval/wer.py`, so this test does exactly what the pipeline does.

In [ ]:
import json, re

def read_json(p): return json.loads(Path(p).read_text(encoding="utf-8"))

ASR_FILTER = 'highpass=f=100,afftdn=nf=-30,atrim=start=0.025,asetpts=PTS-STARTPTS,apad=pad_dur=0.025,loudnorm=I=-20:TP=-2'
EGYPTIAN_STYLE = 'إزيك؟ عامل إيه؟ إحنا النهارده بنتكلم، عشان كده لازم نشوف الموضوع ده دلوقتي، يعني مش كده.'
HALLUCINATIONS = ('ترجمة نانسي', 'نانسي قنقر', 'اشتركوا في القناة', 'اشترك في القناة', 'لا تنسوا الاشتراك', 'شكرا للمشاهدة', 'شكراً للمشاهدة', 'thanks for watching', 'subscribe to', 'subtitles by')
GARBAGE = re.compile('[\u3000-\u9fff가-\ud7af�]')
ARABIC = re.compile('[\u0600-ۿ]')


def hint_words(s: Path, lang):
    """Hint text for Whisper, built only from what we know about THIS session (works for any kind of video)."""
    meta = read_json(s / "session.json")
    up = read_json(s / "upload.json") if (s / "upload.json").exists() else {}
    parts = [EGYPTIAN_STYLE] if lang == "ar" else []
    for text in [up.get("name", ""), *(meta.get("keywords") or up.get("keywords") or [])]:
        if text and (lang != "ar" or ARABIC.search(text)):          # an English title would confuse Arabic
            parts.append(text)
    names = (meta.get("speaker_names") or up.get("speaker_names") or {}).values()
    parts += [n for n in names if n]                                 # helps Whisper spell the names
    return " ".join(parts) or None


def is_hallucination(text):
    low = text.lower()
    return bool(GARBAGE.search(text)) or any(h in low for h in HALLUCINATIONS)


def transcribe(model, audio, lang, hints):
    """The Whisper settings used everywhere (pipeline AND bake-off, so the bake-off measures the real thing)."""
    segments, _ = model.transcribe(audio, language=lang, word_timestamps=True, vad_filter=True, beam_size=5,
                                   condition_on_previous_text=False,     # stops repeated-sentence loops
                                   hotwords=hints,                       # added to EVERY 30 s window
                                   hallucination_silence_threshold=2.0)  # skip text invented over silence
    return [seg for seg in segments if seg.text.strip()]         # list(): Whisper really runs here


def norm_ar(t):
    t = re.sub(r"[\u064B-\u0652\u0640]", "", t)   # remove tashkeel + tatweel
    t = re.sub(r"[إأآ]", "ا", t)
    t = t.replace("ى", "ي").replace("ة", "ه")
    return " ".join(re.sub(r"[^\w\s]", " ", t).split())


HINTS = {s: hint_words(Path(DRIVE_SESSIONS) / s, read_json(Path(DRIVE_SESSIONS) / s / "session.json")["language"])
         for s in SESSIONS}
for s, h in HINTS.items():
    print(s, "hint words:", h)

## Step 5 - prepare the audio (our sessions + MGB-3 clips from the internet)

In [ ]:
import subprocess, numpy as np, soundfile as sf
WORK = Path("/content/work"); WORK.mkdir(parents=True, exist_ok=True)

def ffmpeg(*args):
    subprocess.run(["ffmpeg", "-y", "-v", "error", *args], check=True)

def cut(path):
    x, sr = sf.read(path, dtype="float32")
    return x[int(START * sr):int(END * sr)]

audio, lang = {}, {}
for s in SESSIONS:
    src = str(Path(DRIVE_SESSIONS) / s / "audio.wav")
    ffmpeg("-i", src, "-ac", "1", "-ar", "16000", str(WORK / f"{s}_raw.wav"))
    ffmpeg("-i", src, "-af", ASR_FILTER, "-ac", "1", "-ar", "16000", str(WORK / f"{s}_clean.wav"))  # whole file, like the pipeline
    audio[s, "raw"], audio[s, "clean"] = cut(WORK / f"{s}_raw.wav"), cut(WORK / f"{s}_clean.wav")
    lang[s] = read_json(Path(DRIVE_SESSIONS) / s / "session.json")["language"]

refs = {}
for s in SESSIONS:
    p = Path(RESULTS) / f"ref_{s}.txt"
    if p.exists():
        refs[s] = norm_ar(p.read_text(encoding="utf-8"))
print("hand-corrected text found for:", list(refs) or "none (our sessions get no WER, only confidence)")

In [ ]:
mgb = []   # (audio, correct text)
if MGB3_CLIPS:
    from datasets import load_dataset, Audio
    ds = load_dataset("MightyStudent/Egyptian-ASR-MGB-3", split="train", streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False)).shuffle(seed=0, buffer_size=300)  # mix of many videos
    for i, row in enumerate(ds.take(MGB3_CLIPS)):
        f = WORK / f"mgb_{i}"
        f.write_bytes(row["audio"]["bytes"])
        ffmpeg("-i", str(f), "-ac", "1", "-ar", "16000", f"{f}.wav")
        text = norm_ar(row["sentence"])
        if text:
            mgb.append((sf.read(f"{f}.wav", dtype="float32")[0], text))
    print(len(mgb), "MGB-3 clips,", round(sum(len(a) for a, _ in mgb) / 16000 / 60, 1), "minutes")

## Step 6 - get the models ready
Fine-tuned models from Hugging Face are in a different format, so they are **converted** for faster-whisper.
The converter does not copy two small settings files, so we take them from the original OpenAI model
(without `preprocessor_config.json` faster-whisper would read turbo's audio wrong).

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

def model_path(spec):
    if isinstance(spec, str):
        return spec                                   # faster-whisper downloads it by itself
    hf_id, base = spec
    home = Path(RESULTS) / "models" if KEEP_MODELS_ON_DRIVE else Path("/content/models")
    out = home / hf_id.replace("/", "__")
    if not (out / "model.bin").exists():
        tmp = Path("/content/convert") / out.name
        print("converting", hf_id, "(download + convert, a few minutes) ...")
        subprocess.run(["ct2-transformers-converter", "--model", hf_id, "--output_dir", str(tmp),
                        "--quantization", "float16", "--force"], check=True)
        for f in ["tokenizer.json", "preprocessor_config.json"]:
            shutil.copy(hf_hub_download(base, f), tmp / f)
        shutil.copytree(tmp, out, dirs_exist_ok=True)
        shutil.rmtree(tmp)
    return str(out)

## Step 7 - run the test (one model at a time)

In [ ]:
import gc, time, jiwer, pandas as pd
from faster_whisper import WhisperModel

TEXTS = Path(RESULTS) / "texts"; TEXTS.mkdir(parents=True, exist_ok=True)
rows = []

def score(segs, ref):
    kept = [g for g in segs if not is_hallucination(g.text)]
    text = " ".join(g.text.strip() for g in kept)
    dur = sum(g.end - g.start for g in segs) or 1
    conf = float(np.exp(sum(g.avg_logprob * (g.end - g.start) for g in segs) / dur)) if segs else 0.0
    hyp = norm_ar(text)
    return text, dict(WER=jiwer.wer(ref, hyp) if ref else None, CER=jiwer.cer(ref, hyp) if ref else None,
                      confidence=round(conf, 3), invented=len(segs) - len(kept), words=len(hyp.split()))

for name, spec in MODELS.items():
    print(f"\n=== {name}")
    model = WhisperModel(model_path(spec), device="cuda", compute_type="int8_float16")  # same as the pipeline
    for s in SESSIONS:
        for variant, key, hints in [("raw", "raw", None), ("clean", "clean", None), ("clean+hints", "clean", HINTS[s])]:
            t0 = time.time()
            segs = transcribe(model, audio[s, key], lang[s], hints)
            text, m = score(segs, refs.get(s))
            (TEXTS / f"{s}_{name}_{variant}.txt").write_text(text, encoding="utf-8")
            rows.append(dict(test=f"{s} {START}-{END}s", model=name, audio=variant, **m, seconds=round(time.time() - t0, 1)))
            print(rows[-1])
    if mgb:
        t0, hyps, invented, confs = time.time(), [], 0, []
        for a, ref in mgb:
            segs = transcribe(model, a, "ar", None)
            text, m = score(segs, ref)
            hyps.append(norm_ar(text)); invented += m["invented"]; confs.append(m["confidence"])
        refs_m = [r for _, r in mgb]
        rows.append(dict(test="MGB-3 (internet)", model=name, audio="as is", WER=jiwer.wer(refs_m, hyps),
                         CER=jiwer.cer(refs_m, hyps), confidence=round(float(np.mean(confs)), 3),
                         invented=invented, words=sum(len(h.split()) for h in hyps), seconds=round(time.time() - t0, 1)))
        print(rows[-1])
    del model; gc.collect()

## Step 8 - results

In [ ]:
df = pd.DataFrame(rows)
for c in ["WER", "CER"]:
    df[c] = df[c].astype(float).round(3)
df.to_csv(Path(RESULTS) / "results.csv", index=False)
for test, g in df.groupby("test", sort=False):
    print(f"\n##### {test}")
    print(g.drop(columns="test").sort_values(["WER", "confidence"], ascending=[True, False]).to_string(index=False))
print("\nSaved:", Path(RESULTS) / "results.csv", "and every text in", TEXTS)

## How to choose the winner
1. **MGB-3 WER** (lowest wins) = how well each model knows Egyptian Arabic in general.
2. **Our sessions** = what really matters (noisy far-away phone audio). If you added `ref_s01.txt`, lowest WER wins.
   Otherwise compare **confidence** and read the texts in `RESULTS/texts/` next to the audio.
3. **invented** must be 0. Also look at **raw vs clean**: if raw is better, set `clean_audio: false` in
   `configs/default.yaml`.
4. Note: we do not know the training data of `turbo-dialectal`; if it was trained on MGB-3 its MGB-3 score is too
   good. Our own recordings decide.

Then tell Claude the winner: it will give you the steps to use it in `configs/default.yaml -> audio.asr_models.ar`.